# A minimum training loop on CPU

Prerequisites: Primer-PY, the character N-gram notebook, Chapter 5 shapes, and Chapters 6–7. The manual gradient is an independent expectation; the larger classifier uses a frozen split with four noisy training labels.

In [1]:
from pathlib import Path
import sys, json, math
candidates = [Path.cwd(), *Path.cwd().parents]
root = next((p for p in candidates if (p / "data/part-i/ngram.json").is_file()
             and (p / "src/config/book.mjs").is_file()), None)
if root is None:
    raise FileNotFoundError("book repository boundary not found")
sys.path.insert(0, str(root / "code/part-ii"))
print("Repository fixtures found")

Repository fixtures found


In [2]:
from classifier import one_step, run
step = one_step()
print(json.dumps(step, indent=2))
assert step["gradients"] == {"w": [-0.5, -1.0], "bias": -0.5}
assert math.isclose(step["new_loss"], math.log1p(math.exp(-0.3)), abs_tol=1e-12)

{
  "initial_loss": 0.6931471805599453,
  "gradients": {
    "w": [
      -0.5,
      -1.0
    ],
    "bias": -0.5
  },
  "new_w": [
    0.05,
    0.1
  ],
  "new_bias": 0.05,
  "new_z": 0.3,
  "new_probability": 0.574442516811659,
  "new_loss": 0.5543552444685271,
  "perturb_w0_minus_0_01_loss": 0.6981596805078623,
  "perturb_w0_plus_0_01_loss": 0.6881596805078624
}


## Reproduce the fixed split and selection rule

No test curve is used. The training script selects the first minimum of validation loss, then scores the test set. Rerunning fixed code is reproduction, not fresh evidence.

In [3]:
result = run(write=False)
reference = json.loads((root / "data/part-ii/classifier-run.json").read_text())
assert result["selected_step"] == reference["selected_step"] == 5
assert result["curve"][-1]["train"]["correct"] == 24
assert result["curve"][-1]["validation"]["loss"] > result["selected_validation"]["loss"]
assert math.isclose(result["selected_validation"]["loss"], reference["selected_validation"]["loss"], abs_tol=1e-9)
assert result["test_once_after_selection"]["correct"] == reference["test_once_after_selection"]["correct"]

{
  "environment": {
    "python": "3.12.10",
    "torch": "2.7.0",
    "system": "Darwin",
    "machine": "arm64",
    "device": "cpu",
    "dtype": "float64",
    "threads": 1
  },
  "selected_step": 5,
  "selected_train": {
    "loss": 0.5273627548667736,
    "correct": 17,
    "count": 24,
    "accuracy": 0.7083333333333334
  },
  "selected_validation": {
    "loss": 0.5818593929714245,
    "correct": 37,
    "count": 48,
    "accuracy": 0.7708333333333334
  },
  "test_once_after_selection": {
    "loss": 0.4719049728393296,
    "correct": 38,
    "count": 48,
    "accuracy": 0.7916666666666666
  },
  "elapsed_seconds": 0.12766754115000367,
  "one_step": {
    "initial_loss": 0.6931471805599453,
    "gradients": {
      "w": [
        -0.5,
        -1.0
      ],
      "bias": -0.5
    },
    "new_w": [
      0.05,
      0.1
    ],
    "new_bias": 0.05,
    "new_z": 0.3,
    "new_probability": 0.574442516811659,
    "new_loss": 0.5543552444685271,
    "perturb_w0_minus_0_01_loss": 0

## Diagnose the gap

The final model fits all 24 noisy training labels, while validation likelihood deteriorates. Model selection follows validation loss, not training accuracy. The selected snapshot's test score belongs to these 48 synthetic points; it is not a real-world accuracy estimate.